In [ ]:
from llava.eval.run_llava import eval_model ,load_model, eval_model_wo_loading
from sentence_transformers import SentenceTransformer, util

######
# set this variable according to path where llava weights have been downloaded and the directory where to cache
LLAVA_PATH = "llava_weights"
CACHE_DIR = CACHE_DIR
######

model_path = LLAVA_PATH
prompt = ""
image_file = ""
args = type('Args', (), {
    "model_path": model_path,
    "model_base": None,
    "model_name": "LlavaLlamaForCausalLM",
    "query": prompt,
    "conv_mode": None,
    "image_file": image_file,
    "sep": ",",
    "temperature": 0,
    "top_p": None,
    "num_beams": 1,
    "max_new_tokens": 512 ,
    "cache_dir" : CACHE_DIR
})()


### Size based change

In [ ]:
tokenizer, model, image_processor, context_len = load_model(args)

In [ ]:
def size_based_change(img_path, mask):
    prompt = "What is something of a similar size and structure to the "+mask+" that I can replace the "+mask+" with in the image such that the image maintains the same meaning and is realistic in context of the image? Use as few words as possible"
    
    setattr(args,"query",prompt ) 
    setattr(args,"image_file", img_path) 

    return eval_model_wo_loading(tokenizer, model, image_processor, context_len, args)

mask = "bicycle"
image_file = "../../Diffm_FakeImage_Dataset_5k/CityScapes_256/images/darmstadt_000007_000019.png"
size_based_change(image_file, mask)

##### Automate process

In [ ]:
import json
metadata = json.load(open("../../Diffm_FakeImage_Dataset_5k/metadata_5k.json"))

In [ ]:
metadata.keys()

In [ ]:
suggested_changes = dict()

for img_name in metadata:
    img_path = metadata[img_name]["image_path"]
    # make stored img path compatible with system path
    img_path = "../"+img_path
    img_path = img_path.replace("Diffm_FakeImage_Dataset", "Diffm_FakeImage_Dataset_5k")
    ###

    suggested_changes[img_name] = {"image_path":img_path, "changes":[]}

    for obj_mask in  metadata[img_name]["objects"]:
        mask_name = obj_mask["name"]

        suggested_target = size_based_change(img_path, mask_name)
        print(mask_name , " ---> ", suggested_target)

        suggested_changes[img_name]["changes"].append({"source":mask_name,
                                                       "target": suggested_target})
        

In [ ]:
with open("size_based_change.json", "w") as outfile: 
    json.dump(suggested_changes, outfile)

### Semantic change

In [ ]:
tokenizer, model, image_processor, context_len = load_model(args)

In [ ]:
prompt_5_template = "For an image its semantics are defined by the subjects of the image, the actions they are doing, the objects they are interacting with and the location in which this interaction place\
    Things like the main focus of the image, or the actions they are doing, their location, and emotions they are feeling is a big semantic change. Small changes could be changing the color of objects. \
    What is the detailed caption of the image? What is a change I can make in the caption such that it %s changes the semantics of the image. Remember this change must %s change the meaning of the image \
    How will the caption change after making this change? Make the image more realistic. \
    Make 5 such changes. Give me the original and changed caption in the following format for all of the 5 changes\
    'Original caption: \
    Change:\
    Modified Caption: '"

SIZES = ["significantly and dramatically", "moderately and comparatively"]


In [ ]:
def semantic_change(size, image_file):
    prompt = prompt_5_template%(size,size)
    setattr(args,"query",prompt ) 
    setattr(args,"image_file", image_file) 
    # 
    response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
    return response

In [ ]:
import json
metadata = json.load(open("../../Diffm_FakeImage_Dataset_5k/metadata_5k.json"))

In [ ]:
semantic_changes = dict()

for img_name in metadata:
    img_path = metadata[img_name]["image_path"]
    # make stored img path compatible with system path
    img_path = "../"+img_path
    img_path = img_path.replace("Diffm_FakeImage_Dataset", "Diffm_FakeImage_Dataset_5k")
    ###
    semantic_changes[img_name] = {"image_path":img_path, "changes":[]}

    for size in SIZES:
        semantic_changes["changes"] = semantic_change(size, img_path)


In [ ]:
with open("semantic_based_change.json", "w") as outfile: 
    json.dump(semantic_changes, outfile)

### Examples

In [ ]:
prompt_5_template = "For an image its semantics are defined by the subjects of the image, the actions they are doing, the objects they are interacting with and the location in which this interaction place\
    Things like the main focus of the image, or the actions they are doing, their location, and emotions they are feeling is a big semantic change. Small changes could be changing the color of objects. \
    What is the detailed caption of the image? What is a change I can make in the caption such that it %s changes the semantics of the image. Remember this change must %s change the meaning of the image \
    How will the caption change after making this change? Make the image more realistic. \
    Make 5 such changes. Give me the original and changed caption in the following format for all of the 5 changes\
    'Original caption: \
    Change:\
    Modified Caption: '"

image_file = "../../Diffm_FakeImage_Dataset_5k/CelebAHQ_256/images/14.jpg"

print("SIGNIFICANTLY -------------------------")
size = "significantly and drastically"
prompt = prompt_5_template%(size,size)

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
# response

print("MODERATELY -----------------------------")
size = "moderately and slightly"
prompt = prompt_5_template%(size,size)

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
# response

In [ ]:
prompt_5_template = "For an image its semantics are defined by the subjects of the image, the actions they are doing, the objects they are interacting with and the location in which this interaction place\
    Things like the main focus of the image, or the actions they are doing, their location, and emotions they are feeling is a big semantic change. Small changes could be changing the color of objects. \
    What is the detailed caption of the image? What is a change I can make in the caption such that it %s changes the semantics of the image.\
    How will the caption change after making this change? Make the image more realistic. \
    Make 5 such changes. Give me the original and changed caption in the following format for all of the 5 changes\
    'Original caption: \
    Change:\
    Modified Caption: \
     Remember this change must %s change the meaning of the image '"

image_file = "../../Diffm_FakeImage_Dataset_5k/ADE20K_People_256/images/ADE_train_00024460.jpg"

print("SIGNIFICANTLY -------------------------")
size = "significantly and drastically"
prompt = prompt_5_template%(size,size)

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
# response

print("MODERATELY -----------------------------")
size = "moderately and slightly"
prompt = prompt_5_template%(size,size)

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 

In [ ]:
prompt_5_template = "For an image its semantics are defined by the subjects of the image, the actions they are doing, the objects they are interacting with and the location in which this interaction place\
    Things like the main focus of the image, or the actions they are doing, their location, and emotions they are feeling is a big semantic change. Small changes could be changing the color of objects. \
    What is the detailed caption of the image? What is a change I can make in the caption such that it %s changes the semantics of the image.\
    How will the caption change after making this change? Make the image more realistic. \
    Make 5 such changes. Give me the original and changed caption in the following format for all of the 5 changes\
    'Original caption: \
    Change:\
    Modified Caption: \
     Remember this change must %s change the meaning of the image '"

image_file = "../../Diffm_FakeImage_Dataset_5k/ADE20K_People_256/images/ADE_train_00024549.jpg"

print("MODERATELY -----------------------------")
size = "only slightly"
prompt = prompt_5_template%(size,size)

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 

print("SIGNIFICANTLY -------------------------")
size = "significantly and dramatically"
prompt = prompt_5_template%(size,size)

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
# response



In [ ]:
prompt_5_template = "For an image its semantics are defined by the subjects of the image, the actions they are doing, the objects they are interacting with and the location in which this interaction place\
    Things like the main focus of the image, or the actions they are doing, their location, and emotions they are feeling is a big semantic change. Small changes could be changing the color of objects. \
    What is the detailed caption of the image? What is a change I can make in the caption such that it %s changes the semantics of the image.\
    How will the caption change after making this change? Make the image more realistic. \
    Make 5 such changes. Give me the original and changed caption in the following format for all of the 5 changes\
    'Original caption: \
    Change:\
    Modified Caption: \
     Remember this change must %s change the meaning of the image '"

image_file = "../../Diffm_FakeImage_Dataset_5k/CityScapes_256/images/bremen_000254_000019.png"

print("MODERATELY -----------------------------")
size = "only slightly"
prompt = prompt_5_template%(size,size)

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 

print("SIGNIFICANTLY -------------------------")
size = "significantly and dramatically"
prompt = prompt_5_template%(size,size)

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
# response



In [ ]:
prompt_5_template_small = "For an image its semantics are defined by the subjects of the image, the actions they are doing, the objects they are interacting with and the location in which this interaction place\
    Big and large semantic changes are changing the main focus of the image, their actions, their location, and emotions they are feeling. Slight and small changes arechanging the color of objects, or lighting, or time of the day. \
    What is the detailed caption of the image? What is a change I can make in the caption such that it %s changes the semantics of the image. Remember this change must %s change the meaning of the image \
    How will the caption change after making this change?  \
    Make 5 such unique changes. Give me the original and changed caption in the following format for all of the 5 changes\
    'Original caption: \
    Change:\
    Modified Caption: \
    Remember to make each change unique! Change the image to make it more absurd'"

In [ ]:
size = "slightly"
prompt = prompt_5_template_small%(size,size)
image_file = "../../Diffm_FakeImage_Dataset_5k/HumanParsing_256/images/img_02395.png"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 

In [ ]:
prompt_5_template_small = "For an image its semantics are defined by the subjects of the image, the actions they are doing, the objects they are interacting with and the location in which this interaction place\
    Big and large semantic changes are changing the main focus of the image, their actions, their location, and emotions they are feeling. Slight and small changes arechanging the color of objects, or lighting, or time of the day. \
    What is the detailed caption of the image? What is a change I can make in the caption such that it %s changes the semantics of the image.  \
    How will the caption change after making this change?  \
    Make 5 such unique changes. Give me the original and changed caption in the following format for all of the 5 changes\
    'Original caption: \
    Change:\
    Modified Caption: \
    Remember to make each change unique! Change the image to make it believable. Remember it must change the interpretation of the image!'"

In [ ]:
size = "completely and entirely"
prompt = prompt_5_template_small%(size)
image_file = "../../Diffm_FakeImage_Dataset_5k/HumanParsing_256/images/img_02395.png"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 

#### Semantic change between original and modified caption

In [ ]:
response

In [ ]:
sem_sim_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
prompt_5_template = "For an image its semantics are defined by the subject of the image, the actions they are doing, the objects they are interacting with and the location in which this interaction place\
    Changing things such as the thing in the main focus of the image, or the actions they are doing, their location, and emotions they are feeling is a big semantic change in the image. \
    What is the detailed caption of the image? What is a change I can make in the caption such that it %s changes the semantics of the image. Remember this change must %s change the meaning of the image \
    Make 5 such changes. Give me the original and changed caption in the following format for all of the 5 changes\
    'Original caption: \
    Change:\
    Modified Caption: '"

In [ ]:
size = "moderately"
prompt = prompt_5_template%(size,size)
image_file = "../../Diffm_FakeImage_Dataset_5k/HumanParsing_256/images/img_02395.png"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
cosine_similarity =[]
semantic_change = []
orig_captions = []
modified_line = []
similarity_line = []
lines = [l for l in response.split("\n") if len(l)] 
for line in lines:
    index = line.split(":")[0].lower()
    if index == "original caption":
        orig_captions.append(":".join(line.split(":")[1:]))
    elif index == "modified caption":
        modified_line.append(":".join(line.split(":")[1:]))

orig_embed = sem_sim_model.encode(orig_captions[-1])
modified_emb = sem_sim_model.encode(modified_line)

for j in range(len(modified_emb)):
    val = util.pytorch_cos_sim(orig_embed, modified_emb[j])
    cosine_similarity.append(val.numpy()[0][0].tolist())
    semantic_change.append((1-val).numpy()[0][0].tolist())

print(orig_captions)
z = list(zip(modified_line , semantic_change))
for n in z:
    print(n)

In [ ]:
size = "completely"
prompt = prompt_5_template%(size,size)
image_file = "../../Diffm_FakeImage_Dataset_5k/HumanParsing_256/images/img_02395.png"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
cosine_similarity =[]
semantic_change = []
orig_captions = []
modified_line = []
similarity_line = []
lines = [l for l in response.split("\n") if len(l)] 
for line in lines:
    index = line.split(":")[0].lower()
    if index == "original caption":
        orig_captions.append(":".join(line.split(":")[1:]))
    elif index == "modified caption":
        modified_line.append(":".join(line.split(":")[1:]))

orig_embed = sem_sim_model.encode(orig_captions[-1])
modified_emb = sem_sim_model.encode(modified_line)

for j in range(len(modified_emb)):
    val = util.pytorch_cos_sim(orig_embed, modified_emb[j])
    cosine_similarity.append(val.numpy()[0][0].tolist())
    semantic_change.append((1-val).numpy()[0][0].tolist())

print(orig_captions)
z = list(zip(modified_line , semantic_change))
for n in z:
    print(n)

In [ ]:
size = "drastically"
prompt = prompt_5_template%(size,size) + "\n Remember to change the image drastically"
image_file = "../../Diffm_FakeImage_Dataset_5k/HumanParsing_256/images/img_02395.png"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
cosine_similarity =[]
semantic_change = []
orig_captions = []
modified_line = []
similarity_line = []
lines = [l for l in response.split("\n") if len(l)] 
for line in lines:
    index = line.split(":")[0].lower()
    if index == "original caption":
        orig_captions.append(":".join(line.split(":")[1:]))
    elif index == "modified caption":
        modified_line.append(":".join(line.split(":")[1:]))

orig_embed = sem_sim_model.encode(orig_captions[-1])
modified_emb = sem_sim_model.encode(modified_line)

for j in range(len(modified_emb)):
    val = util.pytorch_cos_sim(orig_embed, modified_emb[j])
    cosine_similarity.append(val.numpy()[0][0].tolist())
    semantic_change.append((1-val).numpy()[0][0].tolist())

print(orig_captions)
z = list(zip(modified_line , semantic_change))
for n in z:
    print(n)

In [ ]:
size = "drastically"
prompt = prompt_5_template%(size,size) + "\n Remember to change the image drastically"
image_file = "../../Diffm_FakeImage_Dataset_5k/CelebAHQ_256/images/14.jpg"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
cosine_similarity =[]
semantic_change = []
orig_captions = []
modified_line = []
similarity_line = []
lines = [l for l in response.split("\n") if len(l)] 
for line in lines:
    index = line.split(":")[0].lower()
    if index == "original caption":
        orig_captions.append(":".join(line.split(":")[1:]))
    elif index == "modified caption":
        modified_line.append(":".join(line.split(":")[1:]))

orig_embed = sem_sim_model.encode(orig_captions[-1])
modified_emb = sem_sim_model.encode(modified_line)

for j in range(len(modified_emb)):
    val = util.pytorch_cos_sim(orig_embed, modified_emb[j])
    cosine_similarity.append(val.numpy()[0][0].tolist())
    semantic_change.append((1-val).numpy()[0][0].tolist())

print(orig_captions)
z = list(zip(modified_line , semantic_change))
for n in z:
    print(n)

In [ ]:
size = "moderately"
prompt = prompt_5_template%(size,size) + "\n Remember to change the image moderately"
image_file = "../../Diffm_FakeImage_Dataset_5k/CelebAHQ_256/images/14.jpg"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
cosine_similarity =[]
semantic_change = []
orig_captions = []
modified_line = []
similarity_line = []
lines = [l for l in response.split("\n") if len(l)] 
for line in lines:
    index = line.split(":")[0].lower()
    if index == "original caption":
        orig_captions.append(":".join(line.split(":")[1:]))
    elif index == "modified caption":
        modified_line.append(":".join(line.split(":")[1:]))

orig_embed = sem_sim_model.encode(orig_captions[-1])
modified_emb = sem_sim_model.encode(modified_line)

for j in range(len(modified_emb)):
    val = util.pytorch_cos_sim(orig_embed, modified_emb[j])
    cosine_similarity.append(val.numpy()[0][0].tolist())
    semantic_change.append((1-val).numpy()[0][0].tolist())

print(orig_captions)
z = list(zip(modified_line , semantic_change))
for n in z:
    print(n)

In [ ]:
size = "drastically"
prompt = prompt_5_template%(size,size) + "\n Remember to change the image drastically"
image_file = "../../Diffm_FakeImage_Dataset_5k/CityScapes_256/images/aachen_000091_000019.png"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
cosine_similarity =[]
semantic_change = []
orig_captions = []
modified_line = []
similarity_line = []
lines = [l for l in response.split("\n") if len(l)] 
for line in lines:
    index = line.split(":")[0].lower()
    if index == "original caption":
        orig_captions.append(":".join(line.split(":")[1:]))
    elif index == "modified caption":
        modified_line.append(":".join(line.split(":")[1:]))

orig_embed = sem_sim_model.encode(orig_captions[-1])
modified_emb = sem_sim_model.encode(modified_line)

for j in range(len(modified_emb)):
    val = util.pytorch_cos_sim(orig_embed, modified_emb[j])
    cosine_similarity.append(val.numpy()[0][0].tolist())
    semantic_change.append((1-val).numpy()[0][0].tolist())

print(orig_captions)
z = list(zip(modified_line , semantic_change))
for n in z:
    print(n)

In [ ]:
size = "slightly"
prompt = prompt_5_template%(size,size) +"\nMake sure to change the image" 
image_file = "../../Diffm_FakeImage_Dataset_5k/CityScapes_256/images/aachen_000091_000019.png"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
cosine_similarity =[]
semantic_change = []
orig_captions = []
modified_line = []
similarity_line = []
lines = [l for l in response.split("\n") if len(l)] 
for line in lines:
    index = line.split(":")[0].lower()
    if index == "original caption":
        orig_captions.append(":".join(line.split(":")[1:]))
    elif index == "modified caption":
        modified_line.append(":".join(line.split(":")[1:]))

orig_embed = sem_sim_model.encode(orig_captions[-1])
modified_emb = sem_sim_model.encode(modified_line)

for j in range(len(modified_emb)):
    val = util.pytorch_cos_sim(orig_embed, modified_emb[j])
    cosine_similarity.append(val.numpy()[0][0].tolist())
    semantic_change.append((1-val).numpy()[0][0].tolist())

print(orig_captions)
z = list(zip(modified_line , semantic_change))
for n in z:
    print(n)

In [ ]:
size = "completely and entirely"
prompt = prompt_5_template%(size,size)
image_file = "../../Diffm_FakeImage_Dataset_5k/HumanParsing_256/images/img_02395.png"

setattr(args,"query",prompt ) 
setattr(args,"image_file", image_file) 
# 
response = eval_model_wo_loading(tokenizer, model, image_processor, context_len, args) 
# response